In [0]:
%run "./data_ingestion"

In [0]:
df_cal.limit(5).display()

Date
2015-01-01
2015-01-02
2015-01-03
2015-01-04
2015-01-05


In [0]:
df_cus.limit(5).display()
df_procat.limit(5).display()
df_sales.limit(5).display()
df_pro.limit(5).display()
df_returns.limit(5).display()

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
11000,MR.,JON,YANG,1966-04-08,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y
11001,MR.,EUGENE,HUANG,1965-05-14,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N
11002,MR.,RUBEN,TORRES,1965-08-12,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
11003,MS.,CHRISTY,ZHU,1968-02-15,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N
11004,MRS.,ELIZABETH,JOHNSON,1968-08-08,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y


ProductCategoryKey,CategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories


OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2016-01-01,2002-10-17,SO48797,385,14335,1,1,1
2016-01-01,2002-09-30,SO48802,383,24923,9,1,1
2016-01-01,2002-11-29,SO48801,326,15493,1,1,1
2016-01-01,2002-11-16,SO48799,352,26708,4,1,1
2016-01-01,2002-12-16,SO48798,369,23332,9,1,1


ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
214,31,HL-U509-R,"Sport-100 Helmet, Red",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Red,0,0,13.0863,34.99
215,31,HL-U509,"Sport-100 Helmet, Black",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Black,0,0,12.0278,33.6442
218,23,SO-B909-M,"Mountain Bike Socks, M",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,M,U,3.3963,9.5
219,23,SO-B909-L,"Mountain Bike Socks, L",Mountain Bike Socks,Combination of natural and synthetic fibers stays dry and provides just the right cushioning.,White,L,U,3.3963,9.5
220,31,HL-U509-B,"Sport-100 Helmet, Blue",Sport-100,"Universal fit, well-vented, lightweight , snap-on visor.",Blue,0,0,12.0278,33.6442


ReturnDate,TerritoryKey,ProductKey,ReturnQuantity
2015-01-18,9,312,1
2015-01-18,10,310,1
2015-01-21,8,346,1
2015-01-22,4,311,1
2015-02-02,6,312,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

##TRANSFORMATION

In [0]:
silver_calendar= (df_cal.withColumn('Date', to_date('Date', 'M/d/yyyy'))\
    .withColumn('Year',year('Date'))\
    .withColumn('Month', month('Date'))\
    .withColumn('MonthName', date_format('Date', 'MMMM'))\
    .withColumn('Quarter', quarter('Date'))\
    .withColumn('DayOfWeek', date_format('Date', 'EEEE'))\
    .withColumn('IsWeekend', dayofweek('Date').isin([1, 7]))\
    .dropDuplicates(['Date']))


In [0]:
silver_calendar.write.format('parquet')\
            .mode('overwrite')\
            .save('/Volumes/retail_sales_lakehouse/default/silver/silver_calendar')

In [0]:
silver_customers = (
    df_cus.withColumn('AnnualIncome', regexp_replace(col('AnnualIncome'), '[$,]', '').cast(DoubleType()))
    .withColumn('BirthDate', to_date('BirthDate', 'M/d/yyyy'))
    .withColumn('Age', floor(datediff(current_date(), col('BirthDate')) / 365.25))
    .withColumn('AgeBucket',
                when(col('Age') < 25, '18-24')
                 .when(col('Age') < 35, '25-34')
                 .when(col('Age') < 45, '35-44')
                 .when(col('Age') < 55, '45-54')
                 .when(col('Age') < 65, '55-64')
                 .otherwise('65+'))
    .withColumn('MaritalStatus', when(col('MaritalStatus') == 'M', 'Married').when(col('MaritalStatus') == 'S', 'Single').otherwise('Unknown'))
    .withColumn('Gender', when(col('Gender') == 'M', 'Male').when(col('Gender') == 'F', 'Female').otherwise('Unknown'))
    .withColumn('FirstName', initcap(trim(col('FirstName'))))
    .withColumn('LastName', initcap(trim(col('LastName'))))
    .withColumn('EmailAddress', lower(trim(col('EmailAddress'))))
    .withColumn('HomeOwner', when(col('HomeOwner') == 'Y', True).otherwise(False))
    .dropDuplicates(['CustomerKey'])
)

In [0]:
silver_customers.limit(5).display()

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,Age,AgeBucket
11016,MR.,Wyatt,Hill,1979-04-28,Married,Male,wyatt32@adventure-works.com,30000.0,0,Partial College,Skilled Manual,true,47,45-54
11033,MR.,Jaime,Nath,1947-09-23,Married,Male,jaime41@adventure-works.com,20000.0,4,High School,Skilled Manual,true,78,65+
11044,MR.,Adam,Flores,1949-05-24,Married,Male,adam10@adventure-works.com,20000.0,2,Partial High School,Clerical,true,77,65+
11052,MRS.,Heidi,Lopez,1951-08-07,Single,Female,heidi19@adventure-works.com,40000.0,2,Partial College,Clerical,false,75,65+
11082,null,Angela,Butler,1966-08-04,Single,Unknown,angela17@adventure-works.com,130000.0,0,Graduate Degree,Management,false,60,55-64


In [0]:
silver_customers.write.format('parquet')\
            .mode('overwrite')\
            .save('/Volumes/retail_sales_lakehouse/default/silver/silver_customers')

In [0]:
silver_product_categories = df_procat.withColumn('CategoryName', trim('CategoryName')).dropDuplicates(['ProductCategoryKey'])
silver_product_subcategories = df_subcat.withColumn('SubcategoryName', trim('SubcategoryName')).dropDuplicates(['ProductSubcategoryKey'])

In [0]:
silver_product_categories.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_product_categories')
silver_product_subcategories.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_product_subcategories')

In [0]:
silver_products = (
    df_pro
    .withColumn('ProductSize', when((col('ProductSize') == '0') | col('ProductSize').isNull(), 'N/A').otherwise(col('ProductSize')))
    .withColumn('ProductStyle', when((col('ProductStyle') == '0') | col('ProductStyle').isNull(), 'N/A').otherwise(col('ProductStyle')))
    .withColumn('ProductCost', round(col('ProductCost').cast(DoubleType()), 2))
    .withColumn('ProductPrice', round(col('ProductPrice').cast(DoubleType()), 2))
    .withColumn('ProfitMargin', round((col('ProductPrice') - col('ProductCost')) / col('ProductPrice'), 4))
    .withColumn('ProductDescription', trim('ProductDescription'))
    .dropDuplicates(['ProductKey'])
)

In [0]:
silver_products.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_products')

In [0]:
df_sales.limit(5).display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2016-01-01,2002-10-17,SO48797,385,14335,1,1,1
2016-01-01,2002-09-30,SO48802,383,24923,9,1,1
2016-01-01,2002-11-29,SO48801,326,15493,1,1,1
2016-01-01,2002-11-16,SO48799,352,26708,4,1,1
2016-01-01,2002-12-16,SO48798,369,23332,9,1,1


In [0]:
silver_sales = (
    df_sales
    .withColumn('OrderDate', to_date('OrderDate', 'M/d/yyyy'))
    .withColumn('StockDate', to_date('StockDate', 'M/d/yyyy'))
    .dropDuplicates(['OrderNumber', 'OrderLineItem'])
    .filter(col('OrderQuantity') > 0)
)

In [0]:
silver_sales.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_sales')

In [0]:
silver_returns = (
    df_returns
    .withColumn('ReturnDate', to_date('ReturnDate', 'M/d/yyyy'))
    .filter(col('ReturnQuantity') > 0)
)

In [0]:
silver_returns.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_returns')

In [0]:
df_ter.limit(5).display()

SalesTerritoryKey,Region,Country,Continent
1,Northwest,United States,North America
2,Northeast,United States,North America
3,Central,United States,North America
4,Southwest,United States,North America
5,Southeast,United States,North America


In [0]:
silver_territories = (
    df_ter
    .withColumn('Region', trim('Region'))
    .withColumn('Country', trim('Country'))
    .withColumn('Continent', trim('Continent'))
    .dropDuplicates(['SalesTerritoryKey'])
)

In [0]:
silver_territories.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/silver/silver_territories')

### Data Quality checkpoint non-existent Keys

In [0]:
foreign_products = silver_sales.select('ProductKey').distinct().join(silver_products.select('ProductKey'), 'ProductKey', 'left_anti').count()
foreign_customers = silver_sales.select('CustomerKey').distinct().join(silver_customers.select('CustomerKey'), 'CustomerKey', 'left_anti').count()
foreign_territories = silver_sales.select('TerritoryKey').distinct().join(silver_territories.select('SalesTerritoryKey').withColumnRenamed('SalesTerritoryKey', 'TerritoryKey'), 'TerritoryKey', 'left_anti').count()
print(f'non-existent ProductKeys: {foreign_products} | non-existent CustomerKeys: {foreign_customers} | non-existent TerritoryKeys: {foreign_territories}')

non-existent ProductKeys: 0 | non-existent CustomerKeys: 0 | non-existent TerritoryKeys: 0


### Gold Layer (Star Schema)

#dim_date


In [0]:
dim_date = silver_calendar.select(
    'Date', 'Year', 'Month', 'MonthName', 'Quarter', 'DayOfWeek', 'IsWeekend'
)

In [0]:
dim_date.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/dim_date')

#dim_customer

In [0]:
dim_customer = silver_customers.select(
    'CustomerKey', 'Prefix', 'FirstName', 'LastName', 'BirthDate', 'Age', 'AgeBucket',
    'MaritalStatus', 'Gender', 'EmailAddress', 'AnnualIncome', 'TotalChildren',
    'EducationLevel', 'Occupation', 'HomeOwner'
)

In [0]:
dim_customer.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/dim_customer')


## DimProduct
##Denormalize category + subcategory into the product dimension — one wide lookup instead of three joins downstream.

In [0]:
dim_product = (
    silver_products
    .join(silver_product_subcategories, 'ProductSubcategoryKey', 'left')
    .join(silver_product_categories, 'ProductCategoryKey', 'left')
    .select(
        'ProductKey', 'ProductSKU', 'ProductName', 'ModelName', 'ProductColor',
        'ProductSize', 'ProductStyle', 'ProductCost', 'ProductPrice', 'ProfitMargin',
        'SubcategoryName', 'CategoryName'
    )
)

In [0]:
dim_product.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/dim_product')


##dim_territory

In [0]:
dim_territory = silver_territories.select('SalesTerritoryKey', 'Region', 'Country', 'Continent')

In [0]:
dim_territory.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/dim_territory')


## FactSales
####Grain = one row per order line. Bring in price/cost from DimProduct to compute Revenue/Cost/Profit 

In [0]:
fact_sales = (
    silver_sales
    .join(dim_product.select('ProductKey', 'ProductPrice', 'ProductCost'), 'ProductKey', 'left')
    .withColumn('Revenue', round(col('OrderQuantity') * col('ProductPrice'), 2))
    .withColumn('Cost', round(col('OrderQuantity') * col('ProductCost'), 2))
    .withColumn('Profit', round(col('Revenue') - col('Cost'), 2))
    .select(
        'OrderNumber', 'OrderLineItem', 'OrderDate', 'StockDate',
        'ProductKey', 'CustomerKey', 'TerritoryKey',
        'OrderQuantity', 'Revenue', 'Cost', 'Profit'
    )
)

In [0]:
fact_sales.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/fact_sales')

##fact returns

In [0]:
fact_returns = silver_returns.select('ReturnDate', 'TerritoryKey', 'ProductKey', 'ReturnQuantity')

In [0]:
fact_returns.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/fact_returns')

##Gold mart

## Monthly revenue & profit by territory and category

In [0]:
mart_monthly_sales_by_territory_category = (
    fact_sales
    .join(dim_territory, fact_sales.TerritoryKey == dim_territory.SalesTerritoryKey, 'left')
    .join(dim_product.select('ProductKey', 'CategoryName'), 'ProductKey', 'left')
    .withColumn('YearMonth', date_format('OrderDate', 'yyyy-MM'))
    .groupBy('YearMonth', 'Region', 'Country', 'CategoryName')
    .agg(
        sum('Revenue').alias('TotalRevenue'),
        sum('Profit').alias('TotalProfit'),
        sum('OrderQuantity').alias('UnitsSold'),
    )
)


In [0]:
mart_monthly_sales_by_territory_category.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/mart_monthly_sales_by_territory_category')

##Return rate by product

In [0]:
units_sold_by_product = fact_sales.groupBy('ProductKey').agg(sum('OrderQuantity').alias('UnitsSold'))
units_returned_by_product = fact_returns.groupBy('ProductKey').agg(sum('ReturnQuantity').alias('UnitsReturned'))

mart_return_rate_by_product = (
    units_sold_by_product
    .join(units_returned_by_product, 'ProductKey', 'left')
    .join(dim_product.select('ProductKey', 'ProductName', 'CategoryName'), 'ProductKey', 'left')
    .fillna(0, subset=['UnitsReturned'])
    .withColumn('ReturnRate', round(col('UnitsReturned') / col('UnitsSold'), 4))
)



In [0]:
mart_return_rate_by_product.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/mart_return_rate_by_product')

###Customer RFM (Recency / Frequency / Monetary) — snapshot as of the max OrderDate in the data

In [0]:
max_date = fact_sales.agg(max('OrderDate')).first()[0]

mart_customer_rfm = (
    fact_sales
    .groupBy('CustomerKey')
    .agg(
        datediff(lit(max_date), max('OrderDate')).alias('RecencyDays'),
        countDistinct('OrderNumber').alias('Frequency'),
        sum('Revenue').alias('monetary')
    )
)

In [0]:
mart_customer_rfm.write.mode('overwrite').format('parquet').save('/Volumes/retail_sales_lakehouse/default/gold/mart_customer_rfm')